# Dataset Inspection: Bone X-ray Tumor Detection

This notebook inspects the two datasets placed under `data/raw/` before any integration work. It preserves each dataset's original terminology and performs no merging, relabeling, preprocessing, resizing, augmentation, SMOTE, or model training.

All counts and conclusions below are based only on files actually found when the notebook is run.

## 1. Imports and project paths

The notebook searches recursively below `data/raw/`. It does not move or modify any original files.

In [ ]:
from collections import Counter
from hashlib import sha256
from pathlib import Path
import os

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw'
FIGURES_ROOT = PROJECT_ROOT / 'results' / 'figures'
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp', '.gif'}
METADATA_EXTENSIONS = {'.csv', '.tsv', '.json', '.xml', '.xlsx', '.xls', '.txt'}
print(f'Project root: {PROJECT_ROOT}')
print(f'Inspecting: {RAW_ROOT}')
print(f'Raw directory exists: {RAW_ROOT.exists()}')

## 2. List everything under `data/raw/`

This first inventory is intentionally broad. Dataset identities are inferred from observed contents later, rather than from fixed folder names.

In [ ]:
all_paths = sorted(RAW_ROOT.rglob('*')) if RAW_ROOT.exists() else []
print(f'Items found: {len(all_paths):,}')
if not all_paths:
    print('No files or folders found. Place both datasets under data/raw/ and rerun the notebook.')
else:
    for path in all_paths:
        kind = 'DIR ' if path.is_dir() else 'FILE'
        print(f'{kind}  {path.relative_to(RAW_ROOT)}')

## 3. Identify dataset roots from their contents

A dataset root is treated as a first-level folder below `data/raw/`. If there are exactly two such folders, both are inspected. If there are more or fewer, the notebook reports that ambiguity and still inspects every first-level folder without guessing which one is BTXRD or Dataset 1.

In [ ]:
def describe_contents(root):
    files = [path for path in root.rglob('*') if path.is_file()]
    images = [path for path in files if path.suffix.lower() in IMAGE_EXTENSIONS]
    csv_files = [path for path in files if path.suffix.lower() == '.csv']
    metadata_files = [path for path in files if path.suffix.lower() in METADATA_EXTENSIONS]
    return {'root': root, 'files': files, 'images': images, 'csv_files': csv_files, 'metadata_files': metadata_files}

dataset_roots = sorted([path for path in RAW_ROOT.iterdir() if path.is_dir()]) if RAW_ROOT.exists() else []
if len(dataset_roots) != 2:
    print(f'Observed {len(dataset_roots)} top-level dataset folders; exactly two were not determinable.')
else:
    print('Two top-level dataset folders found. Their identities will be described from contents, not assumed names.')

dataset_reports = [describe_contents(root) for root in dataset_roots]
for report in dataset_reports:
    print(f'\nDataset candidate: {report["root"]}')
    print(f'Files: {len(report["files"]):,}; images: {len(report["images"]):,}; CSV files: {len(report["csv_files"]):,}; metadata/annotation files: {len(report["metadata_files"]):,}')
    print('Image extensions:', dict(Counter(path.suffix.lower() for path in report['images'])))
    print('Folders:')
    folders = sorted(path for path in report['root'].rglob('*') if path.is_dir())
    print('\n'.join(f'  {path.relative_to(report["root"])}' for path in folders[:200]) or '  None')
    split_names = {'train': [], 'validation': [], 'test': []}
    for path in report['files']:
        lowered = {part.lower() for part in path.parts}
        for split in split_names:
            if split in lowered or (split == 'validation' and 'valid' in lowered):
                split_names[split].append(path)
    print('Split file counts:', {name: len(paths) for name, paths in split_names.items()})

## 4. Inspect CSV, metadata, and annotation files

The code displays columns and five rows for tabular files. It identifies likely image and label columns using column names and value overlap with discovered image filenames. This is a heuristic for inspection only; it does not rewrite labels.

In [ ]:
IMAGE_COLUMN_HINTS = ('image', 'file', 'filename', 'path', 'id')
LABEL_COLUMN_HINTS = ('label', 'class', 'category', 'diagnosis', 'tumor', 'cancer', 'benign', 'malignant', 'normal')

def likely_columns(table, hints):
    return [column for column in table.columns if any(hint in str(column).lower() for hint in hints)]

def inspect_tabular_file(path, images):
    try:
        if path.suffix.lower() == '.csv':
            table = pd.read_csv(path)
        elif path.suffix.lower() == '.tsv':
            table = pd.read_csv(path, sep='\t')
        elif path.suffix.lower() in {'.xlsx', '.xls'}:
            table = pd.read_excel(path)
        elif path.suffix.lower() == '.json':
            table = pd.read_json(path)
        else:
            print('Skipped non-tabular metadata format:', path.name)
            return None
    except Exception as error:
        print(f'Could not read {path}: {error}')
        return None
    print(f'\nFile: {path}')
    print('Columns:', list(table.columns))
    display(table.head(5))
    image_columns = likely_columns(table, IMAGE_COLUMN_HINTS)
    label_columns = likely_columns(table, LABEL_COLUMN_HINTS)
    print('Likely image columns:', image_columns or 'Not determinable from column names')
    print('Likely label columns:', label_columns or 'Not determinable from column names')
    image_names = {image.name for image in images}
    for column in image_columns:
        overlap = table[column].astype(str).map(lambda value: Path(value).name in image_names).sum()
        print(f'Filename overlap for {column!r}: {overlap:,} of {len(table):,} rows')
    for column in label_columns:
        print(f'Value counts for {column!r}:')
        display(table[column].value_counts(dropna=False).rename_axis(column).to_frame('count'))
    return table

tabular_tables = {}
for report in dataset_reports:
    for metadata_path in report['metadata_files']:
        if metadata_path.suffix.lower() in {'.csv', '.tsv', '.xlsx', '.xls', '.json'}:
            tabular_tables[str(metadata_path)] = inspect_tabular_file(metadata_path, report['images'])

## 5. Image properties, readability, and exact duplicates

Images are opened only to read metadata and verify readability. SHA-256 hashes detect exact file duplicates; no duplicate is deleted or altered.

In [ ]:
def image_properties(path):
    try:
        with Image.open(path) as image:
            image.load()
            return {'path': str(path), 'readable': True, 'width': image.width, 'height': image.height, 'mode': image.mode, 'format': image.format, 'error': ''}
    except (OSError, UnidentifiedImageError, ValueError) as error:
        return {'path': str(path), 'readable': False, 'width': np.nan, 'height': np.nan, 'mode': '', 'format': '', 'error': str(error)}

def hash_file(path):
    digest = sha256()
    with path.open('rb') as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

image_details = []
for report in dataset_reports:
    for image_path in report['images']:
        details = image_properties(image_path)
        details['dataset'] = report['root'].name
        details['sha256'] = hash_file(image_path) if details['readable'] else pd.NA
        image_details.append(details)
image_details = pd.DataFrame(image_details)
if image_details.empty:
    print('No image files were available for inspection.')
else:
    print('Image properties sample:')
    display(image_details.head(10))
    print('Unreadable/corrupted image findings:', int((~image_details['readable']).sum()))
    hash_groups = image_details.dropna(subset=['sha256']).groupby('sha256')
    duplicate_groups = hash_groups.filter(lambda group: len(group) > 1)
    print('Exact duplicate image records:', len(duplicate_groups))
    if not duplicate_groups.empty:
        display(duplicate_groups[['dataset', 'path', 'sha256']].sort_values('sha256'))
    else:
        print('No exact duplicate hashes were found.')

## 6. Representative images and saved plots

Representative images are shown with original folder-derived labels when a class folder is present. Labels from CSVs are not guessed or rewritten.

In [ ]:
def original_folder_label(path, root):
    known = {'normal', 'cancer', 'benign', 'malignant', 'tumor'}
    return next((part for part in path.relative_to(root).parts[:-1] if part.lower() in known), 'Label not determinable')

for report in dataset_reports:
    sample = report['images'][: min(6, len(report['images']))]
    if not sample:
        continue
    columns = min(3, len(sample))
    rows = int(np.ceil(len(sample) / columns))
    figure, axes = plt.subplots(rows, columns, figsize=(12, 4 * rows))
    axes = np.atleast_1d(axes).ravel()
    for axis, image_path in zip(axes, sample):
        try:
            with Image.open(image_path) as image:
                axis.imshow(image)
            axis.set_title(original_folder_label(image_path, report['root']))
        except (OSError, UnidentifiedImageError):
            axis.set_title('Unreadable image')
        axis.axis('off')
    for axis in axes[len(sample):]:
        axis.axis('off')
    dataset_name = report['root'].name
    figure.suptitle('Representative images: ' + dataset_name)
    figure.tight_layout()
    figure.savefig(FIGURES_ROOT / (dataset_name + '_representative_images.png'), dpi=150)
    plt.show()

if not image_details.empty:
    format_counts = image_details.groupby(['dataset', 'format']).size().reset_index(name='image_count')
    figure, axis = plt.subplots(figsize=(10, 5))
    for dataset, group in format_counts.groupby('dataset'):
        axis.bar(group['format'].astype(str), group['image_count'], label=dataset, alpha=0.8)
    axis.set_title('Image formats by dataset')
    axis.set_xlabel('Format')
    axis.set_ylabel('Image count')
    axis.legend()
    figure.tight_layout()
    figure.savefig(FIGURES_ROOT / 'dataset_image_formats.png', dpi=150)
    plt.show()

## 7. Dataset comparison table

The table records observed values only. A class is listed from folder names where possible; CSV-derived classes should be reviewed in the preceding section. Train/validation/test counts are file counts whose path contains the corresponding split name.

In [ ]:
comparison_rows = []
for report in dataset_reports:
    details = image_details[image_details['dataset'] == report['root'].name] if not image_details.empty else pd.DataFrame()
    classes = sorted({original_folder_label(path, report['root']) for path in report['images']})
    split_counts = {}
    for split, aliases in {'Train': {'train'}, 'Validation': {'validation', 'valid'}, 'Test': {'test'}}.items():
        split_counts[split] = sum(any(alias in {part.lower() for part in path.parts} for alias in aliases) for path in report['images'])
    resolutions = details[['width', 'height']].dropna().drop_duplicates() if not details.empty else pd.DataFrame()
    typical_resolution = 'Not determinable' if resolutions.empty else '; '.join(f'{int(row.width)}x{int(row.height)}' for _, row in resolutions.head(5).iterrows())
    comparison_rows.append({'Dataset': report['root'].name, 'Total Images': len(report['images']), 'Classes': ', '.join(classes) if classes else 'Not determinable', 'Train': split_counts['Train'], 'Validation': split_counts['Validation'], 'Test': split_counts['Test'], 'Image Formats': ', '.join(sorted({path.suffix.lower() for path in report['images']})) or 'None', 'Typical Resolution': typical_resolution, 'Metadata/Annotations': ', '.join(path.name for path in report['metadata_files']) or 'None'})
comparison_table = pd.DataFrame(comparison_rows, columns=['Dataset', 'Total Images', 'Classes', 'Train', 'Validation', 'Test', 'Image Formats', 'Typical Resolution', 'Metadata/Annotations'])
display(comparison_table)

## Dataset Integration Considerations

This section must be completed from the observed outputs above after the datasets are added. Dataset 1 is expected to use `Cancer` and `Normal`, while BTXRD is expected to use `Normal`, `Benign`, and `Malignant`; those are original terms and are not converted here.

Before integration, review whether the label definitions represent comparable clinical concepts, whether image formats and resolutions differ, whether train/validation/test splits are organized comparably, whether CSV references are complete, and whether corrupted or exact-duplicate files exist. Any relationship between `Benign` and `Cancer` remains undetermined by this inspection and must not be assumed.

In [ ]:
print('DISCOVERY SUMMARY')
print(f'Dataset folders discovered: {len(dataset_reports)}')
for report in dataset_reports:
    details = image_details[image_details['dataset'] == report['root'].name] if not image_details.empty else pd.DataFrame()
    unreadable = int((~details['readable']).sum()) if not details.empty else 0
    dataset_name = report['root'].name
    image_count = len(report['images'])
    csv_count = len(report['csv_files'])
    print(f'- {dataset_name}: {image_count:,} images; {csv_count:,} CSV files; {unreadable:,} unreadable images')
if image_details.empty:
    print('No image counts, classes, formats, or compatibility conclusions can be determined yet.')
else:
    hash_counts = image_details['sha256'].value_counts(dropna=True)
    duplicate_count = int(hash_counts[hash_counts > 1].sum())
    print(f'Exact duplicate image records: {duplicate_count:,}')
print('Decisions required before integration: confirm dataset identity, label semantics, split policy, image compatibility, CSV coverage, and duplicate/corruption handling.')